## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import torch
from IPython.display import Image as IPImage

ROOT        = Path('..')
FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Model Architecture

In [ ]:
from src.model import ConvAutoencoder

model = ConvAutoencoder()
print(model)
print()
print(model.get_model_info())

## 3. Sanity Check — Forward Pass

In [ ]:
from src.dataset import get_dataloaders

loaders = get_dataloaders(ROOT, batch_size=4)
x, _, _ = next(iter(loaders['train']))

model_cpu = ConvAutoencoder()
x_hat, z  = model_cpu(x)

assert x_hat.shape == x.shape, f'Shape mismatch: {x_hat.shape} != {x.shape}'
errors = ConvAutoencoder.reconstruction_error(x, x_hat)

print(f'Input shape:          {tuple(x.shape)}')
print(f'Reconstruction shape: {tuple(x_hat.shape)}')
print(f'Bottleneck shape:     {tuple(z.shape)}')
print(f'Recon error  min={errors.min():.6f}  max={errors.max():.6f}  mean={errors.mean():.6f}')
print('Sanity check passed.')

## 4. Training

> **Note:** Expected ~30 min on RTX 3050. MLflow tracks all metrics.
> Launch the MLflow UI with `mlflow ui` from the project root after training.

In [ ]:
from src.train import train

config = {
    'root_dir':   ROOT,
    'batch_size': 16,
    'lr':         1e-4,
    'epochs':     100,
    'device':     device,
    'patience':   10,
}
train(config)

## 5. Training Results

In [ ]:
curves_path = FIGURES_DIR / 'training_curves.png'
print(f'Loading: {curves_path.resolve()}')
IPImage(filename=str(curves_path))

## 6. Reconstruction Grid

In [ ]:
import matplotlib.pyplot as plt
import torch

from src.dataset import MVTecDataset

# Load best checkpoint into model
checkpoint_path = ROOT / 'models' / 'best_autoencoder.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device).eval()
print(f'Loaded checkpoint: {checkpoint_path.resolve()}')

# Denorm: ImageNet-normalised tensor -> HWC numpy in [0, 1]
_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
def denorm(t):
    return (t.cpu() * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Collect 4 normal + 4 anomalous from test split
test_ds      = MVTecDataset(ROOT, split='test')
normal_idxs  = [i for i, (_, lbl) in enumerate(test_ds.samples) if lbl == 0][:4]
anomaly_idxs = [i for i, (_, lbl) in enumerate(test_ds.samples) if lbl == 1][:4]

# 8 rows x 2 cols: original | reconstruction
fig, axes = plt.subplots(8, 2, figsize=(6, 24))
for row, idx in enumerate(normal_idxs + anomaly_idxs):
    x_s, lbl, fname = test_ds[idx]
    with torch.no_grad():
        x_hat_s, _ = model(x_s.unsqueeze(0).to(device))
    tag = 'normal' if lbl == 0 else 'anomaly'
    err = ConvAutoencoder.reconstruction_error(x_s.unsqueeze(0), x_hat_s.cpu()).item()

    axes[row, 0].imshow(denorm(x_s))
    axes[row, 0].set_title(f'orig [{tag}]  {Path(fname).stem}', fontsize=7)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(denorm(x_hat_s.squeeze(0)))
    axes[row, 1].set_title(f'recon  mse={err:.4f}', fontsize=7)
    axes[row, 1].axis('off')

plt.suptitle('Original | Reconstruction  (top 4 = normal, bottom 4 = anomaly)', fontsize=11)
plt.tight_layout()
out_path = FIGURES_DIR / 'reconstruction_comparison.png'
fig.savefig(out_path, bbox_inches='tight')
print(f'Saved: {out_path.resolve()}')
plt.show()